<a href="https://colab.research.google.com/github/RocioLN15/03_AlgoritmosOptimizacion/blob/main/Trabajo_Pr%C3%A1ctico_Algoritmos_Roc%C3%ADo_Lapeira_Naranjo.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Algoritmos de optimización - Trabajo Práctico<br>
Nombre y Apellidos: Rocío Lapeira Naranjo  <br>
Url GitHub: https://github.com/RocioLN15/03_AlgoritmosOptimizacion <br>
Google Colab: https://colab.research.google.com/drive/13bhLj8wvHGLuORm9HXZ55Q4A4yHqXfcv?usp=sharing <br>
Problema:
>1. Sesiones de doblaje <br>

Descripción del problema: <br>
Se precisa coordinar el doblaje de una película. Los actores del doblaje deben coincidir en las tomas en las que sus personajes aparecen juntos en las diferentes tomas. Los actores de doblaje cobran todos la misma cantidad por cada día que deben desplazarse hasta el estudio de grabación independientemente del número de tomas que se graben. No es posible grabar más de 6 tomas por día.

El objetivo es planificar las sesiones por día de manera que el gasto por los servicios de los actores de doblaje sea el menor posible.

**Los datos son**: <br>
Número de actores: 10 <br>
Número de tomas: 30 <br>                                        

#Modelo
- ¿Cómo represento el espacio de soluciones? <br>
  El espacio de soluciones está formado por un conjunto de individuos representados como un **vector unidimensional de longitud igual al número total de tomas**. En este vector:

  - Cada **gen** corresponde a una toma concreta de la película, identificada por su índice en el vector.
  - El **alelo** almacenado en ese gen es el día asignado para grabar esa toma, expresado como un entero entre 1 y el número total de días disponibles.
  - El **individuo** completo representa una planificación completa de grabación, donde cada toma queda asociada a un día específico.
  
  La **población** es un conjunto de estos individuos, ordenados según su coste total.

- ¿Cuál es la función objetivo? <br>
  La función objetivo consiste en minimizar el coste total del doblaje. Como el único gasto que nos describen es pagar el desplazamiento de los actores, esto es lo mismo que minimizar el número total de actores distintos que deben participar en cada día de grabación.
  
  Para cada día, se identifican las tomas asignadas y se contabilizan los actores que intervienen en ellas. El coste total es la suma, para todos los días, del número de actores distintos presentes, ya que todos cobran lo mismo. Por simplificar el problema, se ha supuesto que el coste por actor y día es 1. El algoritmo genético busca minimizar este valor.
  
  En mi código esa función está implementada en `coste_total(datos, solucion, num_dias)`.

- ¿Cómo implemento las restricciones? <br>
  Hay dos restricciones principales en este problema:
    - La restricción de no repetir tomas, que se cumple de manera implicita por como está representada la solución.
    - La restricción del número máximo de escenas que se pueden grabar cada día, la cual se comprueba con la función `validar_hijo(hijo, num_dias, max_tomas_por_dia)`. Esta función cambia el día de alguna de las tomas que se grababan en días que superaban el máximo.

#Análisis
- ¿Qué complejidad tiene el problema?. Orden de complejidad y Contabilizar el espacio de soluciones <br>
  La combinación de asignación de tomas a días, límite de capacidad por día y minimización de actores distintos por día lo convierte en un problema **NP-hard**, por lo que no existe un algoritmo polinómico conocido que garantice la solución óptima.

  Si se intentara resolver por fuerza bruta **la complejidad sería $O(n!)$**, ya que exploraría todas las soluciones posibles.
  
  El **espacio de soluciones** son todas las asignaciones posibles de tomas a grabar en cada día. Para **contabilizarlo** sería calcular el número total de combinaciones que hay para grabar $T$ tomas diferentes, en $D$ días, sabiendo que solo se pueden grabar $C$ tomas al día, y que no importa el orden de las tomas dentro del día. Por lo que el valor sería: <br>
  <center>$\dfrac{T!}{(C!)^D}$</center>

  En nuestro problema en concreto: <br>
  <center>$\dfrac{30!}{6!^5} = 1.3708...\times 10^{18}$ asignaciones distintas </center>



#Diseño
- ¿Que técnica utilizo? ¿Por qué? <br>
  Debido al tipo de problema que es, NP-hard, y al enorme espacio de soluciones que tiene, no se resulta viable resolverlo con métodos exactos. Por este motivo, es necesario recurrir a técnicas aproximadas.

  Dentro de las técnicas heurísticas y metaheurísticas disponibles he elegido un **algoritmo genético** por varios motivos:

  - Los algoritmos genéticos están diseñados para explorar espacios de búsqueda muy grandes sin necesidad de recorrerlos por completo, lo que los hace apropiados para problemas de planificación y asignación con restricciones.
  - La estructura genética resulta muy sencilla de implementar en este problema: cada toma puede representarse como un gen, el día asignado como alelo y la planificación completa como un individuo.
  - Presenta la capacidad para escapar de óptimos locales, ya que la combinación de operadores de cruce y mutación introduce suficiente variabilidad en la población.
  - El algoritmo permite ajustar parámetros como el tamaño de la población, la tasa de mutación o el número de generaciones, adaptándose tanto a la capacidad computacional disponible como al equilibrio deseado entre calidad de la solución y tiempo de cálculo.

#Código

In [1]:
#Importamos las librerías necesarias
import pandas as pd
import random
import numpy as np
import copy
import math

In [2]:
# Función que crea una solución aleatoria para inicializar el algoritmo
def generar_individuo_aleatorio(num_tomas, max_tomas_por_dia, num_dias):
    individuo = np.full(num_tomas, -1, dtype=int)

    # Crea un array con todas las tomas y las mezcla aleatoriamente
    tomas =  list(range(num_tomas))
    random.shuffle(tomas)

    # Para cada día, elige c tomas (el máximo número de tomas
    # que se pueden grabar en un día) y les asigna ese día
    indice = 0
    for dia in range(1, num_dias+1):
      for _ in range(max_tomas_por_dia):
        if indice >= num_tomas:
          break
        individuo[tomas[indice]] = dia
        indice += 1

    return individuo

# Función que calcula el coste del proyecto para un día en concreto
def coste_dia(datos, tomas):
  suma_actores = np.sum(datos[tomas], axis=0)

  return np.count_nonzero(suma_actores, axis=0)

# Función que calcula el coste total del proyecto
def coste_total(datos, solucion, num_dias):
  solucion = np.array(solucion)

  return sum(coste_dia(datos, np.where(solucion == dia)[0]) for dia in range(1, num_dias+1)) # El np.where obtiene el array de tomas que se graban cada día

# Función que une de forma aleatoria, en una tupla,
# todos los individuos de la población con otro individuo
# para crear los progenitores
def seleccionar_progenitores(poblacion):
    progenitores = []
    for i in range(len(poblacion)):
        progenitor1 = poblacion[i]
        progenitor2 = random.choices(poblacion)[0]
        progenitores.append((progenitor1, progenitor2))
    return progenitores

# Función que cruza dos progenitores. El cruce consiste en elegir un día
# aleatorio y reemplazar el día, en todas las tomas asignadas a ese día en el
# progenitor1, por el día correspondiente en el progenitor2.
def cruzar(progenitor1, progenitor2, num_dias):
    hijo = progenitor1
    dia1 = random.randint(1, num_dias + 1)

    mask = (hijo == dia1)
    indices_dia1 = np.where(mask)[0]

    hijo[mask] = progenitor2[indices_dia1]

    return hijo

# Selecciona dos tomas aleatorias de un individuo y les intercambia
# los días que se graban
def mutar(individuo):
    toma1 = random.randint(0, len(individuo) - 1)
    toma2 = random.randint(0, len(individuo) - 1)
    individuo[toma1], individuo[toma2] = individuo[toma2], individuo[toma1]

# Función que comprueba que el individuo no graba más del máximo
# número de tomas en un día. Si lo hace, cambia el día de una de las tomas,
# elegida aleatoriamente, por el día que menos tomas se graban
def validar_hijo(hijo, num_dias, max_tomas_por_dia):
  tomas_por_dia = np.bincount(hijo, minlength=num_dias+1)[1:]

  while True:
    dia_max = np.argmax(tomas_por_dia) + 1
    dia_min = np.argmin(tomas_por_dia) + 1

    if tomas_por_dia[dia_max - 1] <= max_tomas_por_dia:
      break

    # Obtiene la matriz con las tomas para el día con máximas tomas
    array_tomas = np.where(hijo == dia_max)[0]

    # Genera un índice aleatorio para elegir la toma a cambiar
    indice = random.randint(0, len(array_tomas)-1)

    # Obtiene el índice de esa toma en el hijo
    indice = array_tomas[indice]

    # Cambia el día de esa toma al día con menos carga
    hijo[indice] = dia_min

    # Actualiza contadores tomas por dia
    tomas_por_dia[dia_max - 1] -= 1
    tomas_por_dia[dia_min - 1] += 1


# Función que crea los nuevos individuos mediante el cruce de progenitores
# y la mutación. Además de asegurarse de que cumplan las restricciones
def reproducir(poblacion, max_tomas_por_dia, num_dias):
    # Crea todas las parejas de progenitores
    progenitores = seleccionar_progenitores(poblacion)

    # Realiza los cruces para todos los progenitores
    hijos = [cruzar(copy.deepcopy(progenitor1), copy.deepcopy(progenitor2), num_dias) for progenitor1, progenitor2 in progenitores]

    # Realiza dos mutaciones en los hijos y asegura que sean válidos
    for hijo in hijos:
        if random.random() < 0.1:
            mutar(hijo)
            mutar(hijo)
        validar_hijo(hijo, num_dias, max_tomas_por_dia)
    return hijos

In [3]:
# Datos del problema
datos = np.array([ [1, 1, 1, 1, 1, 0, 0, 0, 0, 0],
                  [0, 0, 1, 1, 1, 0, 0, 0, 0, 0],
                  [0, 1, 0, 0, 1, 0, 1, 0, 0, 0],
                  [1, 1, 0, 0, 0, 0, 1, 1, 0, 0],
                  [0, 1, 0, 1, 0, 0, 0, 1, 0, 0],
                  [1, 1, 0, 1, 1, 0, 0, 0, 0, 0],
                  [1, 1, 0, 1, 1, 0, 0, 0, 0, 0],
                  [1, 1, 0, 0, 0, 1, 0, 0, 0, 0],
                  [1, 1, 0, 1, 0, 0, 0, 0, 0, 0],
                  [1, 1, 0, 0, 0, 1, 0, 0, 1, 0],
                  [1, 1, 1, 0, 1, 0, 0, 1, 0, 0],
                  [1, 1, 1, 1, 0, 1, 0, 0, 0, 0],
                  [1, 0, 0, 1, 1, 0, 0, 0, 0, 0],
                  [1, 0, 1, 0, 0, 1, 0, 0, 0, 0],
                  [1, 1, 0, 0, 0, 0, 1, 0, 0, 0],
                  [0, 0, 0, 1, 0, 0, 0, 0, 0, 1],
                  [1, 0, 1, 0, 0, 0, 0, 0, 0, 0],
                  [0, 0, 1, 0, 0, 1, 0, 0, 0, 0],
                  [1, 0, 1, 0, 0, 0, 0, 0, 0, 0],
                  [1, 0, 1, 1, 1, 0, 0, 0, 0, 0],
                  [0, 0, 0, 0, 0, 1, 0, 1, 0, 0],
                  [1, 1, 1, 1, 0, 0, 0, 0, 0, 0],
                  [1, 0, 1, 0, 0, 0, 0, 0, 0, 0],
                  [0, 0, 1, 0, 0, 1, 0, 0, 0, 0],
                  [1, 1, 0, 1, 0, 0, 0, 0, 0, 1],
                  [1, 0, 1, 0, 1, 0, 0, 0, 1, 0],
                  [0, 0, 0, 1, 1, 0, 0, 0, 0, 0],
                  [1, 0, 0, 1, 0, 0, 0, 0, 0, 0],
                  [1, 0, 0, 0, 1, 1, 0, 0, 0, 0],
                  [1, 0, 0, 1, 0, 0, 0, 0, 0, 0] ])

num_tomas, num_actores = datos.shape
max_tomas_por_dia = 6
num_dias = math.ceil(num_tomas / max_tomas_por_dia)
tam_poblacion = 5000
num_preservar = tam_poblacion // 15
max_generaciones = 500
max_iter_sin_mejora = 60
num_iter_sin_mejora = 0

# Inicializo la población y la ordeno en base a su coste total
poblacion = [generar_individuo_aleatorio(num_tomas, max_tomas_por_dia, num_dias) for _ in range(tam_poblacion)]
poblacion = sorted(poblacion, key=lambda individuo: coste_total(datos, individuo, num_dias))

print("Tamaño población inicial: ", len(poblacion))
print("Numero a preservar: ", num_preservar, "\n")

solucion = poblacion[0]
coste_solucion = coste_total(datos, solucion, num_dias)
for i in range(max_generaciones):
    print("-----------------")
    print(f"GENERACION {i}")
    # Realiza las operaciones para buscar una solución mejor a la anterior
    nueva_poblacion = poblacion[:num_preservar]
    hijos = reproducir(poblacion, max_tomas_por_dia, num_dias)
    hijos_fitness = [1/coste_total(datos, individuo, num_dias) for individuo in hijos]
    nueva_poblacion.extend(random.choices(hijos, weights = hijos_fitness, k = (tam_poblacion - num_preservar)))

    poblacion = sorted(nueva_poblacion, key=lambda individuo: coste_total(datos, individuo, num_dias))

    # Muestra la mejor solución encontrada hasta el momento, su coste y
    # el coste de la solución anterior
    mejor_individuo = poblacion[0]
    coste_mejor_individuo = coste_total(datos, mejor_individuo, num_dias)
    print(f"Mejor individuo: {mejor_individuo}")
    print(f"Coste mejor individuo: {coste_mejor_individuo}")
    print(f"Coste solución anterior: {coste_solucion} \n")

    # Si la nueva solución tiene un mejor coste, guarda la información
    # de la nueva solución y resetea el contador de "no mejora".
    # Si no tiene un mejor coste, aumenta el contador.
    # Si el contador ha llegado al máximo, termina la ejecución
    if coste_solucion > coste_mejor_individuo:
        solucion = copy.deepcopy(mejor_individuo)
        coste_solucion = coste_mejor_individuo
        num_iter_sin_mejora = 0
    else:
        num_iter_sin_mejora += 1
        if num_iter_sin_mejora >= max_iter_sin_mejora:
            break

Tamaño población inicial:  5000
Numero a preservar:  333 

-----------------
GENERACION 0
Mejor individuo: [4 1 4 3 3 2 4 5 5 2 2 5 2 5 3 3 1 1 1 4 1 2 1 5 3 4 3 4 2 5]
Coste mejor individuo: 33
Coste solución anterior: 33 

-----------------
GENERACION 1
Mejor individuo: [4 1 4 3 3 2 4 5 5 2 2 5 2 5 3 3 1 1 1 4 1 2 1 5 3 4 3 4 2 5]
Coste mejor individuo: 33
Coste solución anterior: 33 

-----------------
GENERACION 2
Mejor individuo: [3 1 3 1 1 5 3 5 3 5 1 4 2 2 3 5 2 4 3 1 5 4 4 4 5 1 2 2 2 4]
Coste mejor individuo: 32
Coste solución anterior: 33 

-----------------
GENERACION 3
Mejor individuo: [3 1 3 1 1 5 3 5 3 5 1 4 2 2 3 5 2 4 3 1 5 4 4 4 5 1 2 2 2 4]
Coste mejor individuo: 32
Coste solución anterior: 32 

-----------------
GENERACION 4
Mejor individuo: [3 1 3 1 1 5 3 5 3 5 1 4 2 2 3 5 2 4 3 1 5 4 4 4 5 1 2 2 2 4]
Coste mejor individuo: 32
Coste solución anterior: 32 

-----------------
GENERACION 5
Mejor individuo: [3 1 3 1 1 5 3 5 3 5 1 4 2 2 3 5 2 4 3 1 5 4 4 4 5 1 2 2 2 4]
C